In [4]:
! pip show gymnasium

Name: gymnasium
Version: 1.2.3
Summary: A standard API for reinforcement learning and a diverse set of reference environments (formerly Gym).
Home-page: 
Author: 
Author-email: Farama Foundation <contact@farama.org>
License: MIT License
Location: C:\Users\Soumya\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: cloudpickle, farama-notifications, numpy, typing-extensions
Required-by: 


In [2]:
import gymnasium as gym
import numpy as np
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ModuleNotFoundError: No module named 'gymnasium'

In [ ]:
ENV_NAME = "MountainCar-v0"

def make_env(seed):
    env = gym.make(ENV_NAME)
    env.reset(seed=seed)

    # Override max episode steps to 2000 (IMPORTANT)
    env._max_episode_steps = 2000

    return env

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, d = zip(*batch)

        return (
            torch.FloatTensor(s).to(device),
            torch.LongTensor(a).to(device),
            torch.FloatTensor(r).to(device),
            torch.FloatTensor(s_next).to(device),
            torch.FloatTensor(d).to(device)
        )

    def __len__(self):
        return len(self.buffer)

In [2]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

        self.apply(self.init_weights)

    def init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

NameError: name 'nn' is not defined

In [ ]:
GAMMA = 0.99
LR = 5e-4
BATCH_SIZE = 128
BUFFER_SIZE = 20000

EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 100000   # slow decay important

TARGET_UPDATE = 2000
NUM_EPISODES = 1000
MAX_STEPS = 2000

In [ ]:
def select_action(state, q_net, epsilon, action_dim):
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            return q_net(state).argmax().item()

In [ ]:
def train_dqn(seed=0):
    env = make_env(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    q_net = QNetwork(state_dim, action_dim).to(device)
    target_net = QNetwork(state_dim, action_dim).to(device)
    target_net.load_state_dict(q_net.state_dict())

    optimizer = optim.Adam(q_net.parameters(), lr=LR)
    replay_buffer = ReplayBuffer(BUFFER_SIZE)

    epsilon = EPS_START
    epsilon_decay = (EPS_START - EPS_END) / EPS_DECAY

    rewards = []
    total_steps = 0

    for episode in range(NUM_EPISODES):
        state, _ = env.reset()
        episode_reward = 0

        for step in range(MAX_STEPS):
            total_steps += 1

            action = select_action(state, q_net, epsilon, action_dim)
            next_state, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated

            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward

            # Learning step
            if len(replay_buffer) > BATCH_SIZE:
                s, a, r, s_next, d = replay_buffer.sample(BATCH_SIZE)

                q_values = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    max_next_q = target_net(s_next).max(1)[0]
                    target = r + GAMMA * max_next_q * (1 - d)

                loss = nn.MSELoss()(q_values, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Hard target update
            if total_steps % TARGET_UPDATE == 0:
                target_net.load_state_dict(q_net.state_dict())

            # Epsilon decay
            epsilon = max(EPS_END, epsilon - epsilon_decay)

            if done:
                break

        rewards.append(episode_reward)

        if episode % 10 == 0:
            print(f"Episode {episode}, Reward: {episode_reward}, Epsilon: {epsilon:.3f}")

    env.close()
    return rewards

In [ ]:
rewards = train_dqn(seed=0)

In [ ]:
def moving_average(x, window=20):
    return np.convolve(x, np.ones(window)/window, mode='valid')

plt.plot(rewards, alpha=0.3, label="Raw")
plt.plot(moving_average(rewards), label="Smoothed")
plt.xlabel("Episodes")
plt.ylabel("Return")
plt.title("DQN on MountainCar-v0")
plt.legend()
plt.show()

In [ ]:
all_rewards = []

for seed in range(15):
    rewards = train_dqn(seed)
    all_rewards.append(rewards)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Compute mean and CI ---
all_rewards = np.array(all_rewards)
mean_rewards = np.mean(all_rewards, axis=0)
std = np.std(all_rewards, axis=0)
n = all_rewards.shape[0]

ci = 1.96 * std / np.sqrt(n)

# --- Moving average function ---
def moving_average(x, window=20):
    return np.convolve(x, np.ones(window)/window, mode='valid')

# --- Smooth everything ---
window = 20
smooth_mean = moving_average(mean_rewards, window)
smooth_ci = moving_average(ci, window)

# Adjust x-axis after smoothing
x = np.arange(len(smooth_mean))

# --- Plot ---
plt.figure(figsize=(10, 6))

plt.plot(x, smooth_mean, label="Mean Reward", linewidth=2)

plt.fill_between(
    x,
    smooth_mean - smooth_ci,
    smooth_mean + smooth_ci,
    alpha=0.3,
    label="95% CI"
)

plt.xlabel("Episodes")
plt.ylabel("Total Reward")
plt.title("Training Performance (DQN on MountainCar)")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

NameError: name 'np' is not defined